## Уграђивања

У нашем претходном примеру радили смо са векторима високе димензије "мешанке речи" дужине `vocab_size`, и експлицитно смо вршили конверзију са вектора кратке димензије позицијске репрезентације у ретку, једно-вршну репрезентацију. Ова једно-вршна репрезентација није ефикасна у коришћењу меморије, поред тога, свака реч се третира независно једна од друге, тј. једно-вршно кодирани вектори не изражавају никакву семантичку сличност између речи.

У овој јединици ћемо наставити да истражујемо **News AG** скуп података. За почетак, хајде да учитамо податке и добијемо неке дефиниције из претходне свеске.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Шта је ембединг?

Идеја **ембединга** је да се речи представе нижедимензионалним густим векторима, који на неки начин одражавају семантичко значење речи. Касније ћемо говорити о томе како изградити смислене векторе речи, али за сада размишљајмо о ембедингима као начину смањења димензионалности вектора речи.

Дакле, ембединг слој узима реч као улаз и производи излазни вектор задане дужине `embedding_size`. У једном смислу, врло је сличан слоју `Linear`, али уместо што узима вектор са јединичним кодирањем, може узети број речи као улаз.

Коришћењем ембединг слоја као првог слоја у нашој мрежи, можемо прећи са модела вреће речи на модел **embedding bag**, где прво претварамо сваку реч у нашој речи у одговарајући ембединг, а затим израчунавамо агрегатну функцију преко свих тих ембединга, као што су `sum`, `average` или `max`.

![Слика која приказује класификатор са ембедингом за пет речи у низу.](../../../../../translated_images/sr/embedding-classifier-example.b77f021a7ee67eee.webp)

Наша неуронска мрежа за класификацију почеће са ембединг слојем, затим слојем за агрегацију, и линеарним класификатором на врху:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Рад са променљивом величином секвенце

Као резултат ове архитектуре, минібатцхеви за нашу мрежу морали би бити креирани на одређени начин. У претходној јединици, када смо користили bag-of-words, сви BoW тензори у минібатцху имали су једнаку величину `vocab_size`, без обзира на стварну дужину нашег текстуалног низа. Када пређемо на векторе речи, имали бисмо променљив број речи у сваком текстуалном узорку, и када бисмо те узорке комбиновали у минібатцхеве, морали бисмо применити нешто попуњавање.

Ово се може урадити коришћењем исте технике пружања `collate_fn` функције извору података:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Тренирање класификатора уграђивања

Сада када смо дефинисали исправан дата учитавач, можемо тренирати модел користећи функцију тренинга коју смо дефинисали у претходној јединици:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Напомена**: Овде тренирамо само за 25к записа (мање од једног пуног епоха) због уштеде времена, али можете наставити са тренингом, написати функцију за тренинг током неколико епоха и експериментисати са параметром брзине учења да бисте постигли већу тачност. Требало би да будете у могућности да достигнете тачност од око 90%.


### EmbeddingBag слој и репрезентација секвенци променљиве дужине

У претходној архитектури, морали смо да додајемо паддинг свим секвенцама до исте дужине како бисмо их могли уклопити у minibatch. Ово није најефикаснији начин представљања секвенци променљиве дужине - други приступ би био коришћење **offset** вектора, који би држао офсете свих секвенци смештених у један велики вектор.

![Слика која приказује представљање секвенце помоћу офсета](../../../../../translated_images/sr/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Напомена**: На горњој слици приказујемо секвенцу карактера, али у нашем примеру радимо са секвенцама речи. Међутим, општи принцип представљања секвенци помоћу офсет вектора остаје исти.

За рад са offset репрезентацијом користимо [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) слој. Он је сличан `Embedding`, али прима content вектор и offset вектор као улаз, а такође укључује и просечни слој, који може бити `mean`, `sum` или `max`.

Ево модификоване мреже која користи `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Да бисмо припремили скуп података за тренинг, потребно је да обезбедимо функцију конверзије која ће припремити вектор офсета:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Имајте у виду да, за разлику од свих претходних примера, наша мрежа сада прима два параметра: вектор података и вектор помераја, који су различитих величина. Слично томе, наш учитавач података нам такође пружа 3 вредности уместо 2: и текстуални и вектор помераја се достављају као функције. Због тога морамо мало прилагодити нашу функцију тренинга да би се тиме позабавила:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Семантички уграђаји: Word2Vec

У нашем претходном примеру, слој модела за уграђивање научио је да мапира речи у векторски приказ, али тај приказ није имао много семантичког значења. Било би добро научити такав векторски приказ, да сличне речи или синоними одговарају векторима који су блиски један другом у смислу неке векторске удаљености (нпр. еуклидске удаљености).

Да бисмо то урадили, треба да претходно обучимо наш модел за уграђивање на великој колекцији текста на специфичан начин. Један од првих начина да се обуче семантичко уграђивање зове се [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Заснован је на две главне архитектуре које се користе за продукцију дистрибуиране репрезентације речи:

 - **Континуирани врећа-речи** (CBoW) — у овој архитектури, тренирамо модел да предвиди реч из окружења. Дати нграм $(W_{-2},W_{-1},W_0,W_1,W_2)$, циљ модела је да предвиди $W_0$ из $(W_{-2},W_{-1},W_1,W_2)$.
 - **Континуирани скип-грам** је супротно од CBoW. Модел користи околне речи из прозора контекста да предвиди тренутну реч.

CBoW је бржи, док је скип-грам спорији, али боље представља речи које се ретко јављају.

![Слика која приказује оба алгоритма CBoW и Skip-Gram за претварање речи у векторе.](../../../../../translated_images/sr/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Да бисмо експериментисали са word2vec уграђивањем претходно обученим на скупу података Google News, можемо користити библиотеку **gensim**. Испод налазимо речи најсличније речи 'neural'

> **Напомена:** Када први пут креирате векторе речи, преузимање може потрајати!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Такође можемо израчунати векторске уграђености речи, које ће се користити за тренирање модела класификације (приказујемо само првих 20 компоненти вектора ради јасноће):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Одлична ствар у вези са семантичким уграђивањем је да можете манипулисати векторским кодирањем како бисте променили семантику. На пример, можемо затражити да пронађемо реч чија би векторска репрезентација била што ближа речима *king* и *woman*, а што даља од речи *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

И CBoW и Skip-Grams су „предиктивне“ ембединге, у том смислу што узимају у обзир само локалне контексте. Word2Vec не користи глобални контекст.

**FastText** се надовезује на Word2Vec учењем векторских представљања за сваку реч и карактерске n-граме који се налазе у свакој речи. Вредности представљања се затим просечују у један вектор у сваком кораку тренинга. Иако ово додаје много додатних прорачуна пре тренинга, омогућава да векторска представљања речи кодирају информације испод реч.

Други метод, **GloVe**, користи идеју ко-екстремне матрице, примењује неуронске методе да декомпонује ко-екстремну матрицу у изражајније и нелинеарне векторе речи.

Можете експериментисати са примером мењајући ембединг на FastText и GloVe, јер gensim подржава више различитих модела векторског представљања речи.


## Коришћење предтренираних усађивања у PyTorch-у

Можемо изmenити горе наведен пример да претходно попунимо матрицу у нашем слоју усађивања семантичким усађивањима, као што је Word2Vec. Морамо узети у обзир да речници предтренираних усађивања и нашег текстуалног корпуса вероватно неће бити усклађени, па ћемо тежине за недостајуће речи иницијализовати насумичним вредностима:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Сада хајде да тренирамо наш модел. Имајте у виду да време које је потребно за тренирање модела је знатно дуже него у претходном примеру, због веће величине слоја уграђивања, и тиме много већег броја параметара. Такође, због тога, можда ћемо морати да тренирамо наш модел на више примера ако желимо да избегнемо претеривање прилагођавања.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

У нашем случају не видимо велико повећање тачности, што је вероватно због потпуно различитих вокабулара.
Да бисмо превазишли проблем различитих вокабулара, можемо користити једно од следећих решења:
* Поново тренирати word2vec модел на нашем вокабулару
* Учитајте наш скуп података са вокабуларом из претходно тренираног word2vec модела. Вокабулар који се користи за учитавање скупа података може се одредити током учитавања.

Потња метода делује једноставније, нарочито зато што PyTorch `torchtext` оквир садржи уграђену подршку за ембеддинге. На пример, можемо инстанцирати вокабулар заснован на GloVe на следећи начин:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Учитавање речника има следеће основне операције:
* `vocab.stoi` речник нам омогућава да претворимо реч у њен индекс у речнику
* `vocab.itos` ради супротно - претвара број у реч
* `vocab.vectors` је низ вектора уграђивања, тако да за добијање уграђивања речи `s` треба да користимо `vocab.vectors[vocab.stoi[s]]`

Ево примера манипулације уграђивањима да се покаже једначина **kind-man+woman = queen** (морао сам мало да подесим коефицијент да би функционисало):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Да бисмо обучили класификатор користећи те уграђене вредности, прво морамо да кодирају наш скуп података користећи GloVe вокабулар:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Као што смо горе видели, све векторске вграде се чувају у `vocab.vectors` матрици. Ово чини веома једноставним да се ти тежински параметри учитају у тежине слоја за уграђивање помоћу једноставног копирања:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Хајде сада да тренирамо наш модел и видимо да ли добијамо боље резултате:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Један од разлога зашто не видимо значајно повећање тачности јесте чињеница да неке речи из нашег скупa података недостају у претходно обученом GloVe речнику, и стога се у суштини игноришу. Да бисмо превазишли овај проблем, можемо обучити сопствене уграђене представе на нашем скупу података. 


## Контекстуалне уграђености

Једно кључно ограничење традиционалних претходно тренираних уграђених представљања као што је Word2Vec је проблем дисамбигације значења речи. Док претходно трениране уграђености могу заробити део значења речи у контексту, сва могућа значења речи су кодирана у исту уграђеност. Ово може изазвати проблеме у даљим моделима, јер многе речи као што је реч 'play' имају различита значења у зависности од контекста у коме се користе.

На пример, реч 'play' у овим двема различитим реченицама има прилично различито значење:
- Ишао сам на **представу** у позоришту.
- Џон жели да се **игра** са својим пријатељима.

Претходно трениране уграђености изнад представљају оба ова значења речи 'play' у истој уграђености. Да бисмо превазишли ово ограничење, потребно је да правимо уграђености засноване на **језичком моделу**, који је трениран на великом корпусу текста и *зна* како речи могу бити повезане у различитим контекстима. Разматрање контекстуалних уграђености није обухваћено овим туторијалом, али ћемо се вратити на њих када будемо говорили о језичким моделима у наредном делу.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
